# Features Transacciones VPC por categoria
Replica el patron de tu query POS pero **parametrizado por categoria** (`grupo_final_vpc_n3`) y abriendo por **cash-in / cash-out**.

- Una tabla por categoria: `HM_TRANSACCIONES_<categoria>_VPC` (panel balanceado RUC x periodo).
- Categorias: POS, RECAUDACION, DEPOSITOS, PAGO DE SERVICIOS, PAGOS MASIVOS, PAGOS RECIBIDOS, TRANSFERENCIAS, COBRANZAS, OPERACIONES MESA, REACTIVA.
- Variables por metrica (tot / cash-in / cash-out, monto y trxs): snapshot, lag1, var, ratio, MM3, MM6, MAX3/6, MIN3/6, tendencia, acumulado 12m.

Edita la CONFIG y corre todas las celdas.

In [ ]:
# 1) Instalacion de dependencias
%pip install -q awswrangler

In [ ]:
# 2) Imports y CONFIG
import time
import awswrangler as wr

# ============================ CONFIG (editar aqui) ============================
DATABASE      = "disc_comercial"
WORKGROUP     = "ibk-discovery-comercial-work-group"
S3_OUTPUT     = "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/athena-results/"
PERIODO_INICIO = 202401
PERIODO_FIN    = 202605

# categorias a generar (valores de grupo_final_vpc_n3)
CATEGORIAS = [
    "POS",
    "RECAUDACION",
    "DEPOSITOS",
    "PAGO DE SERVICIOS",
    "PAGOS MASIVOS",
    "PAGOS RECIBIDOS",
    "TRANSFERENCIAS",
    "COBRANZAS",
    "OPERACIONES MESA",
    "REACTIVA",
]
# =============================================================================

In [ ]:
# 3) Utilidades
def suffix(cat: str) -> str:
    """Nombre de tabla seguro: MAYUS, espacios->_, sin acentos."""
    s = cat.upper().replace(" ", "_")
    for a,b in [("Á","A"),("É","E"),("Í","I"),("Ó","O"),("Ú","U"),("Ñ","N")]:
        s = s.replace(a,b)
    return s

def run_ddl(sql: str, label: str = ""):
    qid = wr.athena.start_query_execution(sql=sql, database=DATABASE, workgroup=WORKGROUP, s3_output=S3_OUTPUT)
    res = wr.athena.wait_query(query_execution_id=qid)
    state = res["Status"]["State"]
    if state != "SUCCEEDED":
        raise RuntimeError("[" + label + "] " + state + ": " + str(res["Status"].get("StateChangeReason","")))
    print("  OK [" + label + "] qid=" + qid)
    return qid

def drop_table(name: str):
    run_ddl("DROP TABLE IF EXISTS " + DATABASE + "." + name, "drop " + name)

In [ ]:
# 4) Template SQL (parametrico por categoria)
SQL_TXRS = r"""
-- ============================================================================
-- FEATURES TRANSACCIONES VPC por CATEGORIA - PANEL BALANCEADO (RUC x periodo)
-- Fuente : e_perm_aws.t_agg_mes_vpc_transacc_sav_imp
-- Cruce  : e_perm_aws.T_VPC_CLIENTE_BANCA_FINAL_HST (cuc -> ruc)
-- Abre por CASH-IN / CASH-OUT (universal). Parametrico por {{categoria}}.
-- Una sola corrida por categoria (las ventanas necesitan toda la historia).
-- ============================================================================
-- DROP TABLE IF EXISTS disc_comercial.HM_TRANSACCIONES_{suf}_VPC

CREATE TABLE disc_comercial.HM_TRANSACCIONES_{suf}_VPC
WITH (format='Parquet', parquet_compression='SNAPPY', partitioned_by=ARRAY['periodo']) AS (

WITH params AS (
    SELECT {periodo_inicio} AS periodo_inicio, {periodo_fin} AS periodo_fin
),
-- malla de periodos (sin huecos)
periodos AS (
    SELECT periodo_val FROM (
        SELECT (anio*100 + mes) AS periodo_val
        FROM (SELECT anio FROM UNNEST(SEQUENCE(2024, 2026)) AS t(anio)) a
        CROSS JOIN (SELECT mes FROM UNNEST(SEQUENCE(1, 12)) AS t(mes)) m
    ) g CROSS JOIN params p
    WHERE g.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin
),
-- cuc -> ruc por periodo
cuc_ruc AS (
    SELECT periodo_val, num_ruc_val, cuc_num
    FROM e_perm_aws.T_VPC_CLIENTE_BANCA_FINAL_HST
    CROSS JOIN params
    WHERE fecha_dt IN (
        SELECT MAX(fecha_dt) FROM e_perm_aws.T_VPC_CLIENTE_BANCA_FINAL_HST
        CROSS JOIN params p2
        WHERE CAST(YEAR(fecha_dt)*100 + MONTH(fecha_dt) AS INTEGER) BETWEEN p2.periodo_inicio AND p2.periodo_fin
        GROUP BY CAST(YEAR(fecha_dt)*100 + MONTH(fecha_dt) AS INTEGER)
    )
    AND COALESCE(num_ruc_val,'.') NOT LIKE '.' AND COALESCE(num_ruc_val,'.') NOT LIKE ''
),
-- universo de RUC con la categoria en el rango
universo_ruc AS (
    SELECT DISTINCT r.num_ruc_val
    FROM e_perm_aws.t_agg_mes_vpc_transacc_sav_imp t
    INNER JOIN cuc_ruc r ON t.cuc_num_val = r.cuc_num AND t.periodo_val = r.periodo_val
    CROSS JOIN params p
    WHERE t.grupo_final_vpc_n3 = '{categoria}'
      AND t.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin
      AND r.num_ruc_val IS NOT NULL
),
malla AS (
    SELECT u.num_ruc_val, pe.periodo_val FROM universo_ruc u CROSS JOIN periodos pe
),
-- base con RUC (solo meses con transaccion), trae el sentido cash-in/out
base AS (
    SELECT t.periodo_val, r.num_ruc_val,
           t.tipo_cash_desc AS flujo,
           t.grupo_vpc_desc AS subtipo,
           t.cant_transacc,
           t.montotransaccion_solar_amt AS monto
    FROM e_perm_aws.t_agg_mes_vpc_transacc_sav_imp t
    INNER JOIN cuc_ruc r ON t.cuc_num_val = r.cuc_num AND t.periodo_val = r.periodo_val
    CROSS JOIN params p
    WHERE t.grupo_final_vpc_n3 = '{categoria}'
      AND t.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin
      AND r.num_ruc_val IS NOT NULL
),
-- agregacion por RUC x periodo (cash-in / cash-out)
agg AS (
    SELECT periodo_val, num_ruc_val,
        SUM(monto)                                                          AS tot_monto,
        SUM(cant_transacc)                                                  AS tot_trxs,
        SUM(CASE WHEN flujo='CASH-IN'  THEN monto         ELSE 0 END)       AS cashin_monto,
        SUM(CASE WHEN flujo='CASH-IN'  THEN cant_transacc ELSE 0 END)       AS cashin_trxs,
        SUM(CASE WHEN flujo='CASH-OUT' THEN monto         ELSE 0 END)       AS cashout_monto,
        SUM(CASE WHEN flujo='CASH-OUT' THEN cant_transacc ELSE 0 END)       AS cashout_trxs,
        COUNT(DISTINCT subtipo)                                             AS n_subtipos
    FROM base GROUP BY periodo_val, num_ruc_val
),
-- pivot balanceado (malla LEFT JOIN agg, 0 donde no hubo)
pivot AS (
    SELECT m.num_ruc_val, m.periodo_val,
        COALESCE(a.tot_monto,0)     AS tot_monto,
        COALESCE(a.tot_trxs,0)      AS tot_trxs,
        COALESCE(a.cashin_monto,0)  AS cashin_monto,
        COALESCE(a.cashin_trxs,0)   AS cashin_trxs,
        COALESCE(a.cashout_monto,0) AS cashout_monto,
        COALESCE(a.cashout_trxs,0)  AS cashout_trxs,
        COALESCE(a.n_subtipos,0)    AS n_subtipos,
        CASE WHEN a.num_ruc_val IS NOT NULL THEN 1 ELSE 0 END AS flag_activo_mes
    FROM malla m
    LEFT JOIN agg a ON m.num_ruc_val=a.num_ruc_val AND m.periodo_val=a.periodo_val
),
-- panel con ventanas
panel AS (
    SELECT
        num_ruc_val,
        periodo_val,
        n_subtipos,
        flag_activo_mes,
        SUM(flag_activo_mes) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                        AS meses_activos_3m,
        SUM(flag_activo_mes) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                        AS meses_activos_6m,
        cashin_monto - cashout_monto                          AS flujo_neto_mes,
        cashin_monto / NULLIF(tot_monto,0)                    AS ratio_cashin,

        -- tot_monto
        tot_monto                                                   AS tot_monto_mes,
        LAG(tot_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                                AS tot_monto_lag1,
        tot_monto - LAG(tot_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                          AS tot_monto_var1,
        tot_monto / NULLIF(LAG(tot_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val), 0)               AS tot_monto_ratio1,
        AVG(CAST(tot_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                    AS tot_monto_mm3,
        AVG(CAST(tot_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                    AS tot_monto_mm6,
        MAX(tot_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS tot_monto_max3,
        MAX(tot_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS tot_monto_max6,
        MIN(tot_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS tot_monto_min3,
        MIN(tot_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS tot_monto_min6,
        tot_monto - AVG(CAST(tot_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)              AS tot_monto_tend6,
        SUM(tot_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 11 PRECEDING AND CURRENT ROW)                                   AS tot_monto_acum12,

        -- tot_trxs
        tot_trxs                                                   AS tot_trxs_mes,
        LAG(tot_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                                AS tot_trxs_lag1,
        tot_trxs - LAG(tot_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                          AS tot_trxs_var1,
        tot_trxs / NULLIF(LAG(tot_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val), 0)               AS tot_trxs_ratio1,
        AVG(CAST(tot_trxs AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                    AS tot_trxs_mm3,
        AVG(CAST(tot_trxs AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                    AS tot_trxs_mm6,
        MAX(tot_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS tot_trxs_max3,
        MAX(tot_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS tot_trxs_max6,
        MIN(tot_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS tot_trxs_min3,
        MIN(tot_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS tot_trxs_min6,
        tot_trxs - AVG(CAST(tot_trxs AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)              AS tot_trxs_tend6,
        SUM(tot_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 11 PRECEDING AND CURRENT ROW)                                   AS tot_trxs_acum12,

        -- cashin_monto
        cashin_monto                                                   AS cashin_monto_mes,
        LAG(cashin_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                                AS cashin_monto_lag1,
        cashin_monto - LAG(cashin_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                          AS cashin_monto_var1,
        cashin_monto / NULLIF(LAG(cashin_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val), 0)               AS cashin_monto_ratio1,
        AVG(CAST(cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                    AS cashin_monto_mm3,
        AVG(CAST(cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                    AS cashin_monto_mm6,
        MAX(cashin_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS cashin_monto_max3,
        MAX(cashin_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS cashin_monto_max6,
        MIN(cashin_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS cashin_monto_min3,
        MIN(cashin_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS cashin_monto_min6,
        cashin_monto - AVG(CAST(cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)              AS cashin_monto_tend6,
        SUM(cashin_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 11 PRECEDING AND CURRENT ROW)                                   AS cashin_monto_acum12,

        -- cashin_trxs
        cashin_trxs                                                   AS cashin_trxs_mes,
        LAG(cashin_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                                AS cashin_trxs_lag1,
        cashin_trxs - LAG(cashin_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                          AS cashin_trxs_var1,
        cashin_trxs / NULLIF(LAG(cashin_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val), 0)               AS cashin_trxs_ratio1,
        AVG(CAST(cashin_trxs AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                    AS cashin_trxs_mm3,
        AVG(CAST(cashin_trxs AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                    AS cashin_trxs_mm6,
        MAX(cashin_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS cashin_trxs_max3,
        MAX(cashin_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS cashin_trxs_max6,
        MIN(cashin_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS cashin_trxs_min3,
        MIN(cashin_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS cashin_trxs_min6,
        cashin_trxs - AVG(CAST(cashin_trxs AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)              AS cashin_trxs_tend6,
        SUM(cashin_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 11 PRECEDING AND CURRENT ROW)                                   AS cashin_trxs_acum12,

        -- cashout_monto
        cashout_monto                                                   AS cashout_monto_mes,
        LAG(cashout_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                                AS cashout_monto_lag1,
        cashout_monto - LAG(cashout_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                          AS cashout_monto_var1,
        cashout_monto / NULLIF(LAG(cashout_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val), 0)               AS cashout_monto_ratio1,
        AVG(CAST(cashout_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                    AS cashout_monto_mm3,
        AVG(CAST(cashout_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                    AS cashout_monto_mm6,
        MAX(cashout_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS cashout_monto_max3,
        MAX(cashout_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS cashout_monto_max6,
        MIN(cashout_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS cashout_monto_min3,
        MIN(cashout_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS cashout_monto_min6,
        cashout_monto - AVG(CAST(cashout_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)              AS cashout_monto_tend6,
        SUM(cashout_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 11 PRECEDING AND CURRENT ROW)                                   AS cashout_monto_acum12,

        -- cashout_trxs
        cashout_trxs                                                   AS cashout_trxs_mes,
        LAG(cashout_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                                AS cashout_trxs_lag1,
        cashout_trxs - LAG(cashout_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                          AS cashout_trxs_var1,
        cashout_trxs / NULLIF(LAG(cashout_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val), 0)               AS cashout_trxs_ratio1,
        AVG(CAST(cashout_trxs AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                    AS cashout_trxs_mm3,
        AVG(CAST(cashout_trxs AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                    AS cashout_trxs_mm6,
        MAX(cashout_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS cashout_trxs_max3,
        MAX(cashout_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS cashout_trxs_max6,
        MIN(cashout_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                                    AS cashout_trxs_min3,
        MIN(cashout_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                                    AS cashout_trxs_min6,
        cashout_trxs - AVG(CAST(cashout_trxs AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)              AS cashout_trxs_tend6,
        SUM(cashout_trxs) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 11 PRECEDING AND CURRENT ROW)                                   AS cashout_trxs_acum12
    FROM pivot
)
SELECT
    p.*,
    CAST(p.periodo_val AS VARCHAR) AS periodo
FROM panel p
)
"""

In [ ]:
# 5) Construir una categoria
def build_categoria(categoria: str):
    suf = suffix(categoria)
    tabla = "HM_TRANSACCIONES_" + suf + "_VPC"
    print("[cat] " + categoria + " -> " + tabla)
    drop_table(tabla)
    sql = SQL_TXRS.format(categoria=categoria, suf=suf,
                          periodo_inicio=PERIODO_INICIO, periodo_fin=PERIODO_FIN)
    run_ddl(sql, "create " + suf)

## Ejecutar

In [ ]:
# 6) Correr todas las categorias
t0 = time.time()
for cat in CATEGORIAS:
    build_categoria(cat)
print("Listo en " + str(round((time.time()-t0)/60,1)) + " min. " + str(len(CATEGORIAS)) + " tablas HM_TRANSACCIONES_*_VPC")